<a href="https://colab.research.google.com/github/OutForMilks/Hunger/blob/main/g2p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

**Google Collab**

In [6]:
!git clone https://github.com/OutForMilks/Hunger.git
%cd /content/Hunger

Cloning into 'Hunger'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 32 (delta 4), reused 21 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 35.52 KiB | 17.76 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Hunger


In [1]:
import torch
import torch.nn as nn
import torch_xla.core.xla_model as xm

from pathlib import Path

from src.prep_data import *
from src.decode import *
from src.train import *

ModuleNotFoundError: No module named 'torch_xla'

# **Prepare Data**

In [8]:
input_dir = Path("./data")
output_dir = Path("./prepare")
languages = ["fil"]
dataset_splits = ["train", "dev", "text"]

print("Languages:", " ".join(languages))
for split in dataset_splits:
    print(f"[{split}]")
    convert_split(input_dir, output_dir, languages, split)

Languages: fil
[train]
  [skip] data/fil_train.tsv not found
  -> wrote 0 lines to train.{src,tgt,lang}
[dev]
  [skip] data/fil_dev.tsv not found
  -> wrote 0 lines to dev.{src,tgt,lang}
[text]
  [skip] data/fil_text.tsv not found
  -> wrote 0 lines to text.{src,tgt,lang}


# **Train**

In [ ]:
DATA = Path("./prepare")
OUTPUT = Path("./models")
#seed is a var
STEPS = 200000
SAVE_STEP = 50000

BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

if len(xm.get_xla_supported_devices()) > 0:
    DEVICE = "xla"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 100

cpu


**SEED 1**

In [ ]:
seed = 1
config_1 = CONFIG
config_1["seed"] = seed

if xla:

    import torch_xla.distributed.parallel_loader as pl
    device = xm.xla_device()
    torch.manual_seed(seed)  # xm also seeds RNG on optimizer_step
else:
    device = torch.device(DEVICE)
    torch.manual_seed(seed)

os.makedirs(OUTPUT, exist_ok=True)

src_vocab, tgt_vocab = build_vocabs(
    os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                        os.path.join(DATA, "train.tgt"),
                        src_vocab, tgt_vocab)
max_src, max_tgt = compute_max_lengths(train_ds)
print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
        f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=make_fixed_collate(max_src, max_tgt),
                    drop_last=True)
if xla:
    loader = pl.MpDeviceLoader(loader, device)  # preloads batches onto the TPU

model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                    n_heads=NUM_HEADS, d_ff=FF_SIZE,
                    n_layers=N_LAYERS, dropout=DROPOUT,
                    pad_id=PAD_ID).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

model.train()
data_iter = infinite_loader(loader)
for step in range(1, STEPS + 1):
    batch = next(data_iter)
    if xla:
        src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
    else:
        src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

    logits = model(src, tgt_in, src_pad, tgt_pad)
    loss = criterion(logits, tgt_out)
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

    if step % log_step == 0:
        # .item() forces a host sync; only do it at log intervals on TPU.
        print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                f"lr {opt.param_groups[0]['lr']:.2e}")

    if step % SAVE_STEP == 0 or step == STEPS:
        ckpt = os.path.join(OUTPUT, f"ckpt_{step}.pt")
        payload = {"model": model.state_dict(),
                    "config": config_1,
                    "src_vocab": src_vocab.to_dict(),
                    "tgt_vocab": tgt_vocab.to_dict()}
        if xla:
            xm.save(payload, ckpt)   # moves tensors to CPU; master-only
        else:
            torch.save(payload, ckpt)
        print(f"  saved {ckpt}")

# **Decode**

# **Evaluate**